# Exact SU(3) marked-cluster m4 — one-click production

Press Run All, upload the companion Python file when prompted, and the notebook proceeds automatically into the real exact 609-cluster physical sweep. Resume authentication is managed internally from the frozen script and run label: there is no secret download, password, or hidden prompt. The Drive checkpoint resumes automatically after a disconnect. Set `RUN_COMPLETE_PHYSICAL_SWEEP = False` only for an audit-only preflight.

Runtime truth: use a standard Colab CPU runtime. The exact SymPy/Fraction engine is CPU- and RAM-bound; neither the A100 nor the local 7900 XTX is used. There is no defensible total ETA before representative fresh rows complete. Progress reports exact fresh/resumed counts, and mixed-run ETA is deliberately withheld. The authenticated Drive checkpoint is what makes disconnect/resume survivable. Replaying an older authentic partial checkpoint can lose progress and force recomputation, but it cannot produce PASS without the exact authenticated 609-row manifest and a fresh invocation nonce; true rollback freshness would require an external monotonic anchor and is not claimed here.

In [ ]:
from google.colab import files
from pathlib import Path
from fractions import Fraction
from types import MappingProxyType
import errno, fcntl, getpass, hashlib, hmac, json, os, re, secrets, stat
import shutil, sqlite3, subprocess, sys, tempfile

SCRIPT_NAME = 'DATA_SU3_Exact_MarkedCluster_m4_Colab.py'
EXPECTED_SCRIPT_SHA256 = '9e158af7e7fe012ca452d2455ab6bf6f71b14f5cc5c10a6e49f826b837d32844'
AUTH_MEMFD_NAME = 'hodge_su3_resume_auth_v1'
CERTIFICATE_MEMFD_NAME = 'hodge_su3_certificate_output_v1'
AUTH_SCHEMA = 'HODGE-SU3-RESUME-AUTH-v1'
EXECUTION_ATTESTATION_SCHEMA = 'HODGE-SU3-AUTHENTICATED-EXECUTION-ATTESTATION-v1'
RECOVERY_SECRET_PREFIX = 'HODGE-M4-v1-'
KDF_ITERATIONS = 600_000
KDF_DOMAIN = b'HODGE-SU3-RESUME-PBKDF2-v1\x00'
uploaded = files.upload()
if SCRIPT_NAME not in uploaded:
    raise RuntimeError(f'Upload exactly {SCRIPT_NAME}')
SEALED_SCRIPT_BYTES = bytes(uploaded[SCRIPT_NAME])
observed = hashlib.sha256(SEALED_SCRIPT_BYTES).hexdigest()
if observed != EXPECTED_SCRIPT_SHA256:
    raise RuntimeError(f'Wrong script bytes: {observed} != {EXPECTED_SCRIPT_SHA256}')

def canonical_bytes(value):
    return json.dumps(
        value, sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')

def strict_json_bytes(raw, label):
    if type(raw) is not bytes:
        raise RuntimeError(f'{label} is not bytes')
    def unique_pairs(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise RuntimeError(f'Duplicate key in {label}: {key}')
            result[key] = value
        return result
    def reject_constant(value):
        raise RuntimeError(f'Non-finite value in {label}: {value}')
    try:
        return json.loads(
            raw.decode('utf-8'),
            object_pairs_hook=unique_pairs,
            parse_constant=reject_constant,
        )
    except (UnicodeError, json.JSONDecodeError) as exc:
        raise RuntimeError(f'{label} is not strict UTF-8 JSON') from exc

def require_hex64(value, label):
    if type(value) is not str or re.fullmatch(r'[0-9a-f]{64}', value) is None:
        raise RuntimeError(f'{label} must be exact lowercase 64-hex')
    return value

def resume_hmac(key, domain, payload):
    if type(key) is not bytes or len(key) != 32:
        raise RuntimeError('Resume authentication key is not 32 bytes')
    message = (
        b'HODGE-SU3-RESUME-HMAC-v1\x00'
        + domain.encode('ascii') + b'\x00' + canonical_bytes(payload)
    )
    return hmac.new(key, message, hashlib.sha256).hexdigest()

def derive_resume_key(recovery_secret, salt_hex, run_id, context_sha):
    if (
        type(recovery_secret) is not str
        or not recovery_secret.startswith(RECOVERY_SECRET_PREFIX)
        or len(recovery_secret) != len(RECOVERY_SECRET_PREFIX) + 64
        or re.fullmatch(
            r'[0-9a-f]{64}',
            recovery_secret[len(RECOVERY_SECRET_PREFIX):],
        ) is None
    ):
        raise RuntimeError('Use the generated HODGE-M4-v1- plus 256-bit recovery secret')
    tokens = [
        require_hex64(salt_hex, 'resume salt'),
        require_hex64(run_id, 'resume run id'),
        require_hex64(context_sha, 'authentication context'),
    ]
    salt = KDF_DOMAIN + b''.join(bytes.fromhex(token) for token in tokens)
    return hashlib.pbkdf2_hmac(
        'sha256', recovery_secret.encode('ascii'), salt,
        KDF_ITERATIONS, dklen=32,
    )

def automatic_recovery_secret(run_label):
    if type(run_label) is not str or re.fullmatch(r'[A-Za-z0-9._-]+', run_label) is None:
        raise RuntimeError('RUN_INSTANCE_LABEL must be a simple version label')
    digest = hashlib.sha256(
        b'HODGE-SU3-AUTOMATIC-RESUME-v1\x00'
        + EXPECTED_SCRIPT_SHA256.encode('ascii') + b'\x00'
        + AUTHORIZED_CANDIDATE_MANIFEST_SHA256.encode('ascii') + b'\x00'
        + run_label.encode('ascii')
    ).hexdigest()
    return RECOVERY_SECRET_PREFIX + digest

def seal_bytes(fd, payload, required_seals):
    view = memoryview(payload)
    while view:
        written = os.write(fd, view)
        if written <= 0:
            raise RuntimeError('Short write to sealed in-memory object')
        view = view[written:]
    os.fsync(fd)
    fcntl.fcntl(fd, fcntl.F_ADD_SEALS, required_seals)
    if int(fcntl.fcntl(fd, fcntl.F_GET_SEALS)) != required_seals:
        raise RuntimeError('In-memory object seals are incomplete')
    try:
        os.pwrite(fd, b'x', 0)
    except OSError as exc:
        if exc.errno != errno.EPERM:
            raise
    else:
        raise RuntimeError('Sealed in-memory object remained writable')

def run_verified(
    arguments, *, capture_output=False, auth_bundle=None,
    expect_authenticated_certificate=False,
):
    if hashlib.sha256(SEALED_SCRIPT_BYTES).hexdigest() != EXPECTED_SCRIPT_SHA256:
        raise RuntimeError('In-memory script binding changed')
    if not hasattr(os, 'memfd_create') or not Path('/proc/self/fd').is_dir():
        raise RuntimeError('Linux sealed-memfd execution is required')
    required = (
        fcntl.F_SEAL_WRITE | fcntl.F_SEAL_SHRINK
        | fcntl.F_SEAL_GROW | fcntl.F_SEAL_SEAL
    )
    source_fd = os.memfd_create(
        'hodge_su3_exact_m4', os.MFD_ALLOW_SEALING | os.MFD_CLOEXEC
    )
    auth_fd = None
    output_fd = None
    auth_encoded = None
    try:
        seal_bytes(source_fd, SEALED_SCRIPT_BYTES, required)
        if hashlib.sha256(
            os.pread(source_fd, len(SEALED_SCRIPT_BYTES) + 1, 0)
        ).hexdigest() != EXPECTED_SCRIPT_SHA256:
            raise RuntimeError('Sealed script hash mismatch')
        pass_fds = [source_fd]
        if auth_bundle is not None:
            if set(auth_bundle) != {
                'key', 'resume_salt_hex', 'resume_run_id',
                'authentication_context_sha256', 'invocation_nonce',
            }:
                raise RuntimeError('Authentication bundle fields changed')
            key = auth_bundle['key']
            if type(key) is not bytes or len(key) != 32:
                raise RuntimeError('Derived authentication key is not 32 bytes')
            output_fd = os.memfd_create(
                CERTIFICATE_MEMFD_NAME,
                os.MFD_ALLOW_SEALING | os.MFD_CLOEXEC,
            )
            auth_fd = os.memfd_create(
                AUTH_MEMFD_NAME,
                os.MFD_ALLOW_SEALING | os.MFD_CLOEXEC,
            )
            auth_encoded = canonical_bytes({
                'schema': AUTH_SCHEMA,
                'key_hex': key.hex(),
                'resume_salt_hex': auth_bundle['resume_salt_hex'],
                'resume_run_id': auth_bundle['resume_run_id'],
                'authentication_context_sha256': (
                    auth_bundle['authentication_context_sha256']
                ),
                'invocation_nonce': auth_bundle['invocation_nonce'],
                'certificate_output_fd': output_fd,
            })
            seal_bytes(auth_fd, auth_encoded, required)
            if int(fcntl.fcntl(output_fd, fcntl.F_GET_SEALS)) != 0:
                raise RuntimeError('Certificate output fd was not initially writable')
            pass_fds.extend((auth_fd, output_fd))
        elif expect_authenticated_certificate:
            raise RuntimeError('Authenticated certificate requested without a key bundle')
        env = os.environ.copy()
        for name in tuple(env):
            if name.upper().startswith('PYTHON'):
                del env[name]
        env['HODGE_SU3_M4_SEALED_SOURCE_FD'] = str(source_fd)
        completed = subprocess.run(
            [sys.executable, '-I', '-u', f'/proc/self/fd/{source_fd}', *arguments],
            check=False,
            pass_fds=tuple(pass_fds),
            env=env,
            text=True,
            stdout=(subprocess.PIPE if capture_output else None),
            stderr=(subprocess.STDOUT if capture_output else subprocess.PIPE),
        )
        if completed.returncode != 0:
            detail = completed.stdout if capture_output else completed.stderr
            raise RuntimeError(
                f'Sealed child failed with exit {completed.returncode}:\n{detail}'
            )
        if int(fcntl.fcntl(source_fd, fcntl.F_GET_SEALS)) != required:
            raise RuntimeError('Script seals changed during execution')
        if hashlib.sha256(
            os.pread(source_fd, len(SEALED_SCRIPT_BYTES) + 1, 0)
        ).hexdigest() != EXPECTED_SCRIPT_SHA256:
            raise RuntimeError('Sealed script changed during execution')
        certificate_bytes = None
        if auth_fd is not None:
            if int(fcntl.fcntl(auth_fd, fcntl.F_GET_SEALS)) != required:
                raise RuntimeError('Authentication fd seals changed during execution')
            if hashlib.sha256(
                os.pread(auth_fd, len(auth_encoded) + 1, 0)
            ).digest() != hashlib.sha256(auth_encoded).digest():
                raise RuntimeError('Authentication fd bytes changed during execution')
        if expect_authenticated_certificate:
            if output_fd is None:
                raise RuntimeError('Certificate output fd is missing')
            if int(fcntl.fcntl(output_fd, fcntl.F_GET_SEALS)) != required:
                raise RuntimeError('Child did not irreversibly seal certificate output')
            size = os.fstat(output_fd).st_size
            if size <= 0 or size > (64 << 20):
                raise RuntimeError('Authenticated certificate output size is invalid')
            certificate_bytes = os.pread(output_fd, size + 1, 0)
            if len(certificate_bytes) != size:
                raise RuntimeError('Authenticated certificate output was truncated')
        if capture_output and expect_authenticated_certificate:
            return completed.stdout, certificate_bytes
        if capture_output:
            return completed.stdout
        if expect_authenticated_certificate:
            return certificate_bytes
        return None
    finally:
        auth_encoded = None
        for descriptor in (output_fd, auth_fd, source_fd):
            if descriptor is not None:
                try:
                    os.close(descriptor)
                except OSError:
                    pass

print('UPLOAD SHA256 PASS:', observed)


In [ ]:
run_verified(['--self-test'])


In [ ]:
run_verified(['--show-manifest'])
print('PHASE3_TRIALITY_CANDIDATE_SWEEP_READY_NOT_YET_EVALUATED')
print('NO PHYSICAL CONTRACTION HAS RUN')


## Automatic preflight, resume, and physical sweep

The default Run All path reproduces the reviewed 609-row geometry manifest and then proceeds directly into the real exact calculation. Checkpoint authentication and resume are automatic; there are no credentials to download or enter.

In [ ]:
RUN_COMPLETE_PHYSICAL_SWEEP = True  # set False for audit-only preflight
PERSIST_PHYSICAL_RUN_TO_GOOGLE_DRIVE = False  # simple local run; no Drive dependency
RUN_INSTANCE_LABEL = 'easy-v1'  # change only when intentionally starting a separate run
AUTHORIZED_CLUSTER_EVALUATIONS = 609
AUTHORIZED_CANDIDATE_CERTIFICATE_SHA256 = '4e7f5acfd5610a2bd434e88f94c6ba2ba12a258e618a1249f49472f76c5dbd73'
AUTHORIZED_CANDIDATE_MANIFEST_SHA256 = '40b8bcc72b0b6d310f1b556892190090d6be76cdd82fb6520e385c0ecf9bfcb0'
AUTHORIZED_PREFLIGHT_SHA256 = '576a4a3f00a41f1805fd015836107fb27ebc44190bd57629c13c17cc28e9f16f'
AUTHORIZED_SUPPORT_SHA256 = {
    '0': 'ae4421ca4e8d8a6a6c0ba53b7e31f6c17bb7f24ea6f8411f29a0e28aec3ff77e',
    '1': 'a251fa9e9cdf37ef038d7411c8aff21ee5b1b8ef490abacd46b5e10c346d187d',
    '2': '567af8431fe984ec265384ef1c32fdade92f93e32f5b4e7d8315f195c81bd3a8',
}
AUTHORIZED_NUMERIC_SUPPORT_SHA256 = {
    '0': 'b5fe728a050e90bd4f90e47a4f7f67ed71387b60621548aca241f14b035412a4',
    '1': '99e7006b23cc8246b0c756debfc5c7bcb9ed41cff83eebcdb3f7bfa51f9735d2',
    '2': 'dec80e1ca5fb5ffd5e102aaa82e2a9fb583534f63507abb4fbaf0257c7374025',
}
AUTHORIZED_ROOT_FACE_BY_POL = {'0': 45, '1': 44, '2': 43}
AUTHORIZED_SIZE_HISTOGRAM = {'1': 1, '2': 12, '3': 158, '4': 20, '5': 10, '6': 2}
PREFLIGHT_MARKER = 'TRIALITY_CANDIDATE_PREFLIGHT_PASS_609_NO_PHYSICS'
CURRENT_RUN_VERIFIED_CONSTRUCTION = None

def validate_sqlite_schema(connection):
    integrity = connection.execute('PRAGMA integrity_check(1)').fetchone()
    if integrity != ('ok',):
        raise RuntimeError('Checkpoint SQLite integrity check failed')
    objects = tuple(
        (
            kind, name, table_name,
            None if sql is None else ' '.join(str(sql).split()),
        )
        for kind, name, table_name, sql in connection.execute(
            """SELECT type,name,tbl_name,sql FROM sqlite_master
               WHERE type IN ('table','index','view','trigger')
               ORDER BY type,name"""
        )
    )
    expected = (
        ('index', 'sqlite_autoindex_cluster_results_1', 'cluster_results', None),
        ('index', 'sqlite_autoindex_meta_1', 'meta', None),
        (
            'table', 'cluster_results', 'cluster_results',
            'CREATE TABLE cluster_results ( input_pol INTEGER NOT NULL, '
            'support_json TEXT NOT NULL, gap_json TEXT NOT NULL, '
            'endpoint_certificate_sha256 TEXT NOT NULL, '
            'record_hmac_sha256 TEXT NOT NULL, '
            'PRIMARY KEY (input_pol, support_json) )',
        ),
        (
            'table', 'meta', 'meta',
            'CREATE TABLE meta (key TEXT PRIMARY KEY, value TEXT NOT NULL)',
        ),
    )
    if objects != expected:
        raise RuntimeError('Checkpoint exact SQLite schema changed')
    meta_columns = tuple(
        (row[1], str(row[2]).upper(), row[3], row[5])
        for row in connection.execute('PRAGMA table_info(meta)')
    )
    result_columns = tuple(
        (row[1], str(row[2]).upper(), row[3], row[5])
        for row in connection.execute('PRAGMA table_info(cluster_results)')
    )
    if meta_columns != (
        ('key', 'TEXT', 0, 1), ('value', 'TEXT', 1, 0),
    ) or result_columns != (
        ('input_pol', 'INTEGER', 1, 1),
        ('support_json', 'TEXT', 1, 2),
        ('gap_json', 'TEXT', 1, 0),
        ('endpoint_certificate_sha256', 'TEXT', 1, 0),
        ('record_hmac_sha256', 'TEXT', 1, 0),
    ):
        raise RuntimeError('Checkpoint exact column schema changed')

def read_checkpoint_metadata_without_mutation(path):
    journal = path.with_name(path.name + '-journal')
    wal = path.with_name(path.name + '-wal')
    shm = path.with_name(path.name + '-shm')
    if wal.exists() or shm.exists():
        raise RuntimeError('Authenticated DELETE-journal DB has WAL/SHM sidecars')
    if journal.exists():
        journal_info = os.lstat(journal)
        if (
            journal.is_symlink()
            or not os.path.isfile(journal)
            or journal_info.st_nlink != 1
            or journal_info.st_size > (512 << 20)
        ):
            raise RuntimeError('Checkpoint rollback journal is not a private regular file')
        with tempfile.TemporaryDirectory(prefix='hodge-m4-notebook-recovery-') as temp:
            copied = Path(temp) / path.name
            shutil.copyfile(path, copied)
            shutil.copyfile(journal, copied.with_name(copied.name + '-journal'))
            connection = sqlite3.connect(str(copied), timeout=60.0)
            try:
                validate_sqlite_schema(connection)
                rows = tuple(connection.execute('SELECT key,value FROM meta'))
            finally:
                connection.close()
    else:
        connection = sqlite3.connect(
            path.resolve().as_uri() + '?mode=ro', uri=True, timeout=60.0
        )
        try:
            validate_sqlite_schema(connection)
            rows = tuple(connection.execute('SELECT key,value FROM meta'))
        finally:
            connection.close()
    if any(type(key) is not str or type(value) is not str for key, value in rows):
        raise RuntimeError('Checkpoint metadata is not exact text')
    metadata = dict(rows)
    expected_keys = {
        'schema', 'configuration_sha256', 'runtime_script_sha256',
        'candidate_manifest_sha256', 'authentication_context_sha256',
        'resume_salt_hex', 'resume_run_id', 'key_verifier_hmac_sha256',
        'final_manifest_row_count', 'final_manifest_sha256',
        'final_manifest_hmac_sha256',
    }
    if len(metadata) != len(rows) or set(metadata) != expected_keys:
        raise RuntimeError(
            'Existing checkpoint is not authenticated v3; choose a NEW RUN_INSTANCE_LABEL'
        )
    return metadata

def fraction_token(token):
    if type(token) is not str or token.count('/') != 1:
        raise RuntimeError('Exact checkpoint value is not a rational token')
    left, right = token.split('/', 1)
    if re.fullmatch(r'-?(0|[1-9][0-9]*)', left) is None:
        raise RuntimeError('Rational numerator is not canonical')
    if re.fullmatch(r'[1-9][0-9]*', right) is None:
        raise RuntimeError('Rational denominator is not canonical positive')
    value = Fraction(int(left), int(right))
    if f'{value.numerator}/{value.denominator}' != token:
        raise RuntimeError('Rational token is not reduced/canonical')
    return value

def read_exact_regular_bytes(path, expected_size, label, maximum_size):
    if (
        type(expected_size) is not int or expected_size < 1
        or expected_size > maximum_size
    ):
        raise RuntimeError(f'{label} expected size is invalid')
    before = os.lstat(path)
    if (
        not stat.S_ISREG(before.st_mode) or before.st_nlink != 1
        or before.st_size != expected_size
    ):
        raise RuntimeError(f'{label} is not the exact private regular file')
    flags = os.O_RDONLY | getattr(os, 'O_NONBLOCK', 0)
    if hasattr(os, 'O_NOFOLLOW'):
        flags |= os.O_NOFOLLOW
    descriptor = os.open(path, flags)
    try:
        opened = os.fstat(descriptor)
        if (
            not stat.S_ISREG(opened.st_mode) or opened.st_nlink != 1
            or opened.st_size != expected_size
            or (opened.st_dev, opened.st_ino) != (before.st_dev, before.st_ino)
        ):
            raise RuntimeError(f'{label} changed during bounded open')
        chunks = []
        remaining = expected_size
        while remaining:
            chunk = os.read(descriptor, min(remaining, 1 << 20))
            if not chunk:
                raise RuntimeError(f'{label} was truncated during read')
            chunks.append(chunk)
            remaining -= len(chunk)
        if os.read(descriptor, 1):
            raise RuntimeError(f'{label} grew during bounded read')
        return b''.join(chunks)
    finally:
        os.close(descriptor)

def verify_authenticated_result(
    certificate_bytes, certificate_path, checkpoint_path, key,
    salt_hex, run_id, auth_context, invocation_nonce,
):
    named_certificate_bytes = read_exact_regular_bytes(
        certificate_path, len(certificate_bytes),
        'named construction certificate', 64 << 20,
    )
    if named_certificate_bytes != certificate_bytes:
        raise RuntimeError(
            'Named certificate differs from authenticated child-output bytes'
        )
    certificate = strict_json_bytes(certificate_bytes, 'construction certificate')
    if canonical_bytes(certificate) != certificate_bytes:
        raise RuntimeError('Construction certificate is not canonical JSON')
    certificate_keys = {
        'schema', 'status', 'coefficient', 'construction_sha256',
        'scientific_result_sha256', 'construction_payload',
        'authenticated_execution_attestation', 'gates',
        'lower_coefficients', 'gamma_by_order',
        'endpoint_certificate_sha256', 'runtime_script_sha256',
        'checkpoint', 'persistent_checkpoint', 'candidate_filter_scope',
        'candidate_manifest_sha256', 'candidate_coverage_certificate',
        'physical_cluster_evaluations', 'fresh_cluster_evaluations',
        'resumed_cluster_evaluations', 'authenticated_resume_manifest',
        'invocation_nonce', 'target_inputs', 'certificate_hmac_sha256',
    }
    if type(certificate) is not dict or set(certificate) != certificate_keys:
        raise RuntimeError('Construction certificate exact schema changed')
    certificate_hmac = require_hex64(
        certificate['certificate_hmac_sha256'], 'certificate HMAC'
    )
    unsigned_certificate = dict(certificate)
    del unsigned_certificate['certificate_hmac_sha256']
    expected_certificate_hmac = resume_hmac(
        key, 'final-construction-certificate', unsigned_certificate
    )
    if not hmac.compare_digest(certificate_hmac, expected_certificate_hmac):
        raise RuntimeError('Construction certificate HMAC failed')
    if (
        certificate['schema']
        != 'HODGE-SU3-EXACT-MARKED-CLUSTER-M4-v3-AUTHENTICATED'
        or certificate['status'] != 'PASS_TARGET_BLIND_M4_SEALED'
        or certificate['runtime_script_sha256'] != EXPECTED_SCRIPT_SHA256
        or certificate['physical_cluster_evaluations'] != 609
        or certificate['candidate_filter_scope'] != 'necessary_not_sufficient'
        or certificate['candidate_manifest_sha256']
        != AUTHORIZED_CANDIDATE_MANIFEST_SHA256
        or certificate['invocation_nonce'] != invocation_nonce
        or certificate['target_inputs'] != []
        or certificate['persistent_checkpoint'] != str(checkpoint_path)
    ):
        raise RuntimeError('Construction certificate fixed gates changed')
    coverage = certificate['candidate_coverage_certificate']
    coverage_keys = {
        'schema', 'necessary_not_sufficient', 'manifest_sha256',
        'base_concrete_support_count', 'physical_cluster_evaluation_count',
        'proper_incidence_per_polarization',
        'per_polarization_support_sha256', 'rotations',
        'physics_contractions_run', 'certificate_sha256',
    }
    if type(coverage) is not dict or set(coverage) != coverage_keys:
        raise RuntimeError('Candidate coverage certificate schema changed')
    coverage_unsigned = dict(coverage)
    coverage_digest = coverage_unsigned.pop('certificate_sha256')
    if (
        coverage_digest != AUTHORIZED_CANDIDATE_CERTIFICATE_SHA256
        or hashlib.sha256(canonical_bytes(coverage_unsigned)).hexdigest()
        != coverage_digest
        or coverage['manifest_sha256'] != AUTHORIZED_CANDIDATE_MANIFEST_SHA256
        or coverage['physical_cluster_evaluation_count'] != 609
        or coverage['base_concrete_support_count'] != 203
        or coverage['proper_incidence_per_polarization'] != 724
        or coverage['physics_contractions_run'] != 0
        or coverage['necessary_not_sufficient'] is not True
        or coverage['per_polarization_support_sha256'] != AUTHORIZED_SUPPORT_SHA256
    ):
        raise RuntimeError('Candidate coverage certificate did not reproduce')
    construction = certificate['construction_payload']
    construction_keys = {
        'schema', 'coefficient', 'lower_coefficients', 'gamma_by_order',
        'raw_gap_ledger',
        'mobius_by_channel_sha256', 'coverage_authorities',
        'candidate_manifest_sha256', 'candidate_filter_scope',
        'physical_cluster_evaluations', 'endpoint_certificate_sha256',
        'runtime_script_sha256', 'gates', 'target_inputs',
    }
    if type(construction) is not dict or set(construction) != construction_keys:
        raise RuntimeError('Scientific construction payload schema changed')
    scientific_sha = hashlib.sha256(canonical_bytes(construction)).hexdigest()
    if (
        construction['schema'] != 'HODGE-SU3-TARGET-BLIND-SCIENTIFIC-RESULT-v3'
        or scientific_sha != certificate['construction_sha256']
        or scientific_sha != certificate['scientific_result_sha256']
        or construction['coefficient'] != certificate['coefficient']
        or construction['lower_coefficients'] != certificate['lower_coefficients']
        or construction['gamma_by_order'] != certificate['gamma_by_order']
        or construction['gates'] != certificate['gates']
        or construction['candidate_manifest_sha256']
        != AUTHORIZED_CANDIDATE_MANIFEST_SHA256
        or construction['candidate_filter_scope'] != 'necessary_not_sufficient'
        or construction['physical_cluster_evaluations'] != 609
        or construction['runtime_script_sha256'] != EXPECTED_SCRIPT_SHA256
        or construction['endpoint_certificate_sha256']
        != certificate['endpoint_certificate_sha256']
        or construction['target_inputs'] != []
    ):
        raise RuntimeError('Scientific result SHA or mirrored construction fields failed')
    expected_scientific_gates = [
        'exact-full-t1-physical-clusters',
        'six-face-direct-seven-face-folded-support-bound',
        'translated-intermediate-endpoint-convolution',
        'literal-rooted-mobius-all-three-polarizations',
        'sealed-203-row-candidate-keyset-per-polarization',
        'stage0-triality-filter-only-no-stage1-amplitudes',
        'lower-orders-one-through-three-exact',
        'target-blind-construction',
    ]
    if construction['gates'] != expected_scientific_gates:
        raise RuntimeError('Scientific construction gate manifest changed')
    if construction['coverage_authorities'] != {
        '0': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
        '1': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
        '2': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
    }:
        raise RuntimeError('Scientific coverage-authority map changed')
    raw_gap_ledger = construction['raw_gap_ledger']
    raw_gap_ledger_keys = {
        'schema', 'candidate_manifest_sha256', 'row_count',
        'roots_by_input_polarization',
        'per_polarization_support_sha256',
        'per_polarization_support_count',
        'per_polarization_proper_incidence_count',
        'per_polarization_size_histogram', 'embedding_multiplicity', 'rows',
    }
    if (
        type(raw_gap_ledger) is not dict
        or set(raw_gap_ledger) != raw_gap_ledger_keys
        or raw_gap_ledger['schema'] != 'HODGE-SU3-PHASE3-RAW-GAP-LEDGER-v1'
        or raw_gap_ledger['candidate_manifest_sha256']
        != AUTHORIZED_CANDIDATE_MANIFEST_SHA256
        or raw_gap_ledger['row_count'] != 609
        or type(raw_gap_ledger['embedding_multiplicity']) is not int
        or raw_gap_ledger['embedding_multiplicity'] != 1
        or type(raw_gap_ledger['rows']) is not list
        or len(raw_gap_ledger['rows']) != 609
        or any(
            type(row) is not dict
            or set(row) != {'input_pol', 'support', 'gap_by_order'}
            or type(row['input_pol']) is not int
            or row['input_pol'] not in (0, 1, 2)
            or type(row['support']) is not list
            or any(type(face) is not int for face in row['support'])
            or type(row['gap_by_order']) is not list
            for row in raw_gap_ledger['rows']
        )
    ):
        raise RuntimeError('Scientific raw gap ledger schema changed')
    for field in (
        'roots_by_input_polarization', 'per_polarization_support_count',
        'per_polarization_proper_incidence_count',
    ):
        mapping = raw_gap_ledger[field]
        if (
            type(mapping) is not dict or set(mapping) != {'0', '1', '2'}
            or any(type(value) is not int for value in mapping.values())
        ):
            raise RuntimeError(f'Raw gap ledger {field} integer schema changed')
    histograms = raw_gap_ledger['per_polarization_size_histogram']
    if (
        type(histograms) is not dict or set(histograms) != {'0', '1', '2'}
        or any(
            type(histogram) is not dict
            or set(histogram) != {'1', '2', '3', '4', '5', '6'}
            or any(type(value) is not int for value in histogram.values())
            for histogram in histograms.values()
        )
    ):
        raise RuntimeError('Raw gap ledger histogram schema changed')
    require_hex64(construction['mobius_by_channel_sha256'], 'Möbius ledger SHA')
    require_hex64(
        construction['endpoint_certificate_sha256'], 'endpoint chain SHA'
    )
    expected_lower = (
        Fraction(1), Fraction(11, 306), Fraction(-109151, 249696)
    )
    lower_tokens = construction['lower_coefficients']
    if type(lower_tokens) is not list or len(lower_tokens) != 3:
        raise RuntimeError('Scientific lower-order coefficient schema changed')
    lower_exact = tuple(fraction_token(token) for token in lower_tokens)
    if lower_exact != expected_lower:
        raise RuntimeError('Scientific lower-order exact gates changed')
    gamma_tokens = construction['gamma_by_order']
    if (
        type(gamma_tokens) is not list or len(gamma_tokens) != 4
        or any(
            type(matrix) is not list or len(matrix) != 3
            or any(type(row) is not list or len(row) != 3 for row in matrix)
            for matrix in gamma_tokens
        )
    ):
        raise RuntimeError('Scientific Gamma tensor is not exact 4x3x3')
    gamma_scalars = []
    gamma_exact = []
    for order, matrix in enumerate(gamma_tokens, start=1):
        exact_matrix = [
            [fraction_token(token) for token in row] for row in matrix
        ]
        gamma_exact.append(exact_matrix)
        if any(
            exact_matrix[row][column] != 0
            for row in range(3) for column in range(3) if row != column
        ):
            raise RuntimeError(f'Gamma order {order} has nonzero off-diagonal')
        diagonal = tuple(exact_matrix[index][index] for index in range(3))
        if len(set(diagonal)) != 1:
            raise RuntimeError(f'Gamma order {order} is not cubic/scalar')
        gamma_scalars.append(diagonal[0])
    coefficient_exact = fraction_token(construction['coefficient'])
    if tuple(gamma_scalars[:3]) != expected_lower or gamma_scalars[3] != coefficient_exact:
        raise RuntimeError('Gamma diagonals disagree with lower orders or sealed m4')
    construction_text = canonical_bytes(construction).decode('utf-8').lower()
    for forbidden in (
        'resume_run', 'invocation_nonce', 'hmac', 'recovery_secret',
        'resume_salt', 'fresh_cluster', 'resumed_cluster', 'checkpoint',
    ):
        if forbidden in construction_text:
            raise RuntimeError('Scientific identity contains run-specific authentication data')
    checkpoint = certificate['checkpoint']
    if (
        type(checkpoint) is not dict
        or set(checkpoint) != {
            'status', 'event_count', 'last_event_sha256', 'last_stage'
        }
        or checkpoint['status']
        != 'PHASE3_FULL_T1_MOBIUS_COMPLETE'
        or checkpoint['event_count'] != 1219
        or checkpoint['last_stage'] != 'phase3-full-t1-mobius-complete'
    ):
        raise RuntimeError('Construction checkpoint completion schema changed')
    require_hex64(checkpoint['last_event_sha256'], 'checkpoint event digest')

    if checkpoint_path.with_name(checkpoint_path.name + '-journal').exists():
        raise RuntimeError('Successful child left a rollback journal')
    checkpoint_info = os.lstat(checkpoint_path)
    if (
        not stat.S_ISREG(checkpoint_info.st_mode)
        or checkpoint_info.st_nlink != 1
        or checkpoint_info.st_size < 1
        or checkpoint_info.st_size > (512 << 20)
    ):
        raise RuntimeError('Checkpoint is not a bounded private regular file')
    connection = sqlite3.connect(
        checkpoint_path.resolve().as_uri() + '?mode=ro', uri=True, timeout=60.0
    )
    try:
        validate_sqlite_schema(connection)
        meta_rows = tuple(connection.execute('SELECT key,value FROM meta'))
        metadata = dict(meta_rows)
        expected_meta_keys = {
            'schema', 'configuration_sha256', 'runtime_script_sha256',
            'candidate_manifest_sha256', 'authentication_context_sha256',
            'resume_salt_hex', 'resume_run_id', 'key_verifier_hmac_sha256',
            'final_manifest_row_count', 'final_manifest_sha256',
            'final_manifest_hmac_sha256',
        }
        if len(metadata) != len(meta_rows) or set(metadata) != expected_meta_keys:
            raise RuntimeError('Checkpoint metadata exact schema changed')
        immutable_meta = {
            'schema': 'HODGE-SU3-PHASE3-AUTHENTICATED-SQLITE-v3',
            'configuration_sha256': require_hex64(
                metadata['configuration_sha256'], 'configuration SHA'
            ),
            'runtime_script_sha256': EXPECTED_SCRIPT_SHA256,
            'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
            'authentication_context_sha256': auth_context,
            'resume_salt_hex': salt_hex,
            'resume_run_id': run_id,
        }
        if any(metadata[key] != value for key, value in immutable_meta.items()):
            raise RuntimeError('Checkpoint immutable authentication context changed')
        expected_header_hmac = resume_hmac(
            key, 'checkpoint-header-verifier', immutable_meta
        )
        if not hmac.compare_digest(
            metadata['key_verifier_hmac_sha256'], expected_header_hmac
        ):
            raise RuntimeError('Checkpoint header recovery-secret HMAC failed')
        db_rows = tuple(connection.execute(
            """SELECT input_pol,support_json,gap_json,
                      endpoint_certificate_sha256,record_hmac_sha256
                 FROM cluster_results ORDER BY input_pol,support_json"""
        ))
    finally:
        connection.close()
    if len(db_rows) != 609:
        raise RuntimeError('Authenticated checkpoint is not exactly 609 rows')
    entries = []
    authenticated_gap_rows = []
    endpoint_sha_by_key = {}
    supports_by_pol = {0: [], 1: [], 2: []}
    per_pol_count = {0: 0, 1: 0, 2: 0}
    for pol, support_json, gap_json, endpoint_sha, record_hmac in db_rows:
        if (
            type(pol) is not int or pol not in per_pol_count
            or type(support_json) is not str
            or type(gap_json) is not str
            or type(endpoint_sha) is not str
            or type(record_hmac) is not str
        ):
            raise RuntimeError('Checkpoint row SQLite types changed')
        support = strict_json_bytes(
            support_json.encode('utf-8'), 'checkpoint support'
        )
        if (
            type(support) is not list or not support
            or any(type(face) is not int or face < 0 for face in support)
            or support != sorted(set(support))
            or canonical_bytes(support).decode('utf-8') != support_json
        ):
            raise RuntimeError('Checkpoint support JSON is not canonical')
        gaps = strict_json_bytes(gap_json.encode('utf-8'), 'checkpoint gap row')
        if (
            type(gaps) is not list or len(gaps) != 4
            or any(type(row) is not list or len(row) != 3 for row in gaps)
        ):
            raise RuntimeError('Checkpoint gap row shape changed')
        for row in gaps:
            for token in row:
                fraction_token(token)
        if canonical_bytes(gaps).decode('utf-8') != gap_json:
            raise RuntimeError('Checkpoint gap JSON is not canonical')
        require_hex64(endpoint_sha, 'endpoint certificate SHA')
        require_hex64(record_hmac, 'record HMAC')
        record_payload = {
            'schema': 'HODGE-SU3-PHASE3-AUTHENTICATED-SQLITE-v3',
            'resume_run_id': run_id,
            'authentication_context_sha256': auth_context,
            'configuration_sha256': immutable_meta['configuration_sha256'],
            'runtime_script_sha256': EXPECTED_SCRIPT_SHA256,
            'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
            'input_pol': pol,
            'support_json': support_json,
            'gap_json': gap_json,
            'endpoint_certificate_sha256': endpoint_sha,
        }
        expected_row_hmac = resume_hmac(key, 'cluster-result', record_payload)
        if not hmac.compare_digest(record_hmac, expected_row_hmac):
            raise RuntimeError('Checkpoint row HMAC failed')
        entries.append({
            'input_pol': pol,
            'support_json': support_json,
            'record_hmac_sha256': record_hmac,
        })
        authenticated_gap_rows.append({
            'input_pol': pol,
            'support': support,
            'gap_by_order': gaps,
        })
        supports_by_pol[pol].append(tuple(support))
        endpoint_sha_by_key[(pol, tuple(support))] = endpoint_sha
        per_pol_count[pol] += 1
    if per_pol_count != {0: 203, 1: 203, 2: 203}:
        raise RuntimeError('Checkpoint polarization row census changed')
    if entries != sorted(entries, key=lambda row: (
        row['input_pol'], row['support_json']
    )):
        raise RuntimeError('Checkpoint row manifest order changed')

    reviewed_rows = sorted(
        authenticated_gap_rows,
        key=lambda row: (
            row['input_pol'], len(row['support']), tuple(row['support'])
        ),
    )
    endpoint_chain = '0' * 64
    for row in reviewed_rows:
        endpoint_chain = hashlib.sha256((
            endpoint_chain
            + endpoint_sha_by_key[(row['input_pol'], tuple(row['support']))]
        ).encode('ascii')).hexdigest()
    if (
        endpoint_chain != construction['endpoint_certificate_sha256']
        or endpoint_chain != certificate['endpoint_certificate_sha256']
    ):
        raise RuntimeError('Authenticated endpoint certificate chain changed')
    reviewed_support_sha = {}
    reviewed_support_count = {}
    reviewed_incidence_count = {}
    reviewed_size_histogram = {}
    for pol in range(3):
        ordered_supports = sorted(
            set(supports_by_pol[pol]), key=lambda support: (len(support), support)
        )
        if len(ordered_supports) != 203:
            raise RuntimeError('Numeric candidate support keyset is not 203 unique rows')
        root = AUTHORIZED_ROOT_FACE_BY_POL[str(pol)]
        if (
            any(root not in support or len(support) > 6 for support in ordered_supports)
            or set.intersection(*(set(support) for support in ordered_supports)) != {root}
        ):
            raise RuntimeError('Numeric candidate supports changed root/size scope')
        histogram = {
            str(size): sum(len(support) == size for support in ordered_supports)
            for size in range(1, 7)
        }
        incidence_count = sum(
            set(child) < set(parent)
            for parent in ordered_supports for child in ordered_supports
        )
        digest = hashlib.sha256()
        for support in ordered_supports:
            encoded_support = canonical_bytes(list(support))
            digest.update(len(encoded_support).to_bytes(8, 'big'))
            digest.update(encoded_support)
        if (
            histogram != AUTHORIZED_SIZE_HISTOGRAM
            or incidence_count != 724
            or digest.hexdigest() != AUTHORIZED_NUMERIC_SUPPORT_SHA256[str(pol)]
        ):
            raise RuntimeError('Numeric candidate support identity/incidence changed')
        reviewed_support_sha[str(pol)] = digest.hexdigest()
        reviewed_support_count[str(pol)] = len(ordered_supports)
        reviewed_incidence_count[str(pol)] = incidence_count
        reviewed_size_histogram[str(pol)] = histogram
    expected_raw_gap_ledger = {
        'schema': 'HODGE-SU3-PHASE3-RAW-GAP-LEDGER-v1',
        'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
        'row_count': 609,
        'roots_by_input_polarization': AUTHORIZED_ROOT_FACE_BY_POL,
        'per_polarization_support_sha256': reviewed_support_sha,
        'per_polarization_support_count': reviewed_support_count,
        'per_polarization_proper_incidence_count': reviewed_incidence_count,
        'per_polarization_size_histogram': reviewed_size_histogram,
        'embedding_multiplicity': 1,
        'rows': reviewed_rows,
    }
    if raw_gap_ledger != expected_raw_gap_ledger:
        raise RuntimeError('Sealed raw gap ledger differs from authenticated rows')

    computed_gamma = [
        [[Fraction(0) for _output in range(3)] for _input in range(3)]
        for _order in range(4)
    ]
    reconstructed_mobius = {}
    rows_by_pol = {
        pol: [row for row in reviewed_rows if row['input_pol'] == pol]
        for pol in range(3)
    }
    for pol in range(3):
        for order in range(4):
            for output_pol in range(3):
                omega = {}
                for row in rows_by_pol[pol]:
                    support = tuple(row['support'])
                    value = fraction_token(row['gap_by_order'][order][output_pol])
                    support_set = set(support)
                    value -= sum(
                        child_value for child, child_value in omega.items()
                        if set(child) < support_set
                    )
                    omega[support] = value
                computed_gamma[order][pol][output_pol] = sum(
                    omega.values(), Fraction(0)
                )
                reconstructed_mobius[str((order + 1, pol, output_pol))] = {
                    str(support): f'{value.numerator}/{value.denominator}'
                    for support, value in omega.items()
                }
    if computed_gamma != gamma_exact:
        raise RuntimeError('Authenticated raw rows do not reproduce sealed Gamma')
    reconstructed_mobius_sha = hashlib.sha256(
        canonical_bytes(reconstructed_mobius)
    ).hexdigest()
    if reconstructed_mobius_sha != construction['mobius_by_channel_sha256']:
        raise RuntimeError('Authenticated raw rows do not reproduce Möbius digest')
    resume_manifest = certificate['authenticated_resume_manifest']
    resume_keys = {
        'schema', 'configuration_sha256', 'runtime_script_sha256',
        'authentication_context_sha256', 'resume_run_id',
        'candidate_manifest_sha256', 'row_count', 'entries',
        'manifest_sha256', 'manifest_hmac_sha256',
        'row_manifest_sha256', 'row_manifest_hmac_sha256',
        'invocation_nonce', 'current_run_sha256',
        'fresh_cluster_evaluations', 'resumed_cluster_evaluations',
        'current_run_hmac_sha256',
    }
    if type(resume_manifest) is not dict or set(resume_manifest) != resume_keys:
        raise RuntimeError('Authenticated resume manifest exact schema changed')
    manifest_payload = {
        'schema': 'HODGE-SU3-AUTHENTICATED-ROW-MANIFEST-v1',
        'configuration_sha256': immutable_meta['configuration_sha256'],
        'runtime_script_sha256': EXPECTED_SCRIPT_SHA256,
        'authentication_context_sha256': auth_context,
        'resume_run_id': run_id,
        'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
        'row_count': 609,
        'entries': entries,
    }
    manifest_sha = hashlib.sha256(canonical_bytes(manifest_payload)).hexdigest()
    manifest_hmac = resume_hmac(key, 'final-row-manifest', manifest_payload)
    fresh_count = resume_manifest['fresh_cluster_evaluations']
    resumed_count = resume_manifest['resumed_cluster_evaluations']
    if (
        type(fresh_count) is not int or type(resumed_count) is not int
        or fresh_count < 0 or resumed_count < 0
        or fresh_count + resumed_count != 609
    ):
        raise RuntimeError('Fresh/resumed authenticated counts are invalid')
    current_payload = {
        'schema': EXECUTION_ATTESTATION_SCHEMA,
        'configuration_sha256': immutable_meta['configuration_sha256'],
        'runtime_script_sha256': EXPECTED_SCRIPT_SHA256,
        'authentication_context_sha256': auth_context,
        'resume_run_id': run_id,
        'invocation_nonce': invocation_nonce,
        'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
        'row_manifest_sha256': manifest_sha,
        'row_manifest_hmac_sha256': manifest_hmac,
        'fresh_cluster_evaluations': fresh_count,
        'resumed_cluster_evaluations': resumed_count,
    }
    current_sha = hashlib.sha256(canonical_bytes(current_payload)).hexdigest()
    current_hmac = resume_hmac(
        key, 'current-run-resume-summary', current_payload
    )
    expected_resume = {
        **manifest_payload,
        'manifest_sha256': manifest_sha,
        'manifest_hmac_sha256': manifest_hmac,
        'row_manifest_sha256': manifest_sha,
        'row_manifest_hmac_sha256': manifest_hmac,
        'invocation_nonce': invocation_nonce,
        'fresh_cluster_evaluations': fresh_count,
        'resumed_cluster_evaluations': resumed_count,
        'current_run_sha256': current_sha,
        'current_run_hmac_sha256': current_hmac,
    }
    if resume_manifest != expected_resume:
        raise RuntimeError('Authenticated resume manifest did not reproduce')
    if (
        metadata['final_manifest_row_count'] != '609'
        or metadata['final_manifest_sha256'] != manifest_sha
        or not hmac.compare_digest(
            metadata['final_manifest_hmac_sha256'], manifest_hmac
        )
    ):
        raise RuntimeError('SQLite final authenticated manifest metadata changed')
    execution = certificate['authenticated_execution_attestation']
    expected_execution = {
        'schema': EXECUTION_ATTESTATION_SCHEMA,
        'configuration_sha256': immutable_meta['configuration_sha256'],
        'runtime_script_sha256': EXPECTED_SCRIPT_SHA256,
        'authentication_context_sha256': auth_context,
        'resume_run_id': run_id,
        'invocation_nonce': invocation_nonce,
        'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
        'row_count': 609,
        'row_manifest_sha256': manifest_sha,
        'row_manifest_hmac_sha256': manifest_hmac,
        'fresh_cluster_evaluations': fresh_count,
        'resumed_cluster_evaluations': resumed_count,
        'current_run_sha256': current_sha,
        'current_run_hmac_sha256': current_hmac,
        'gates': [
            'authenticated-checkpoint-header',
            'authenticated-609-row-manifest',
            'fresh-invocation-nonce',
            'authenticated-current-run-summary',
        ],
    }
    if execution != expected_execution:
        raise RuntimeError('Authenticated execution attestation did not reproduce')
    if (
        certificate['fresh_cluster_evaluations'] != fresh_count
        or certificate['resumed_cluster_evaluations'] != resumed_count
    ):
        raise RuntimeError('Certificate fresh/resumed counts changed')
    coefficient = fraction_token(certificate['coefficient'])
    if coefficient != coefficient_exact:
        raise RuntimeError('Top-level m4 differs from scientific Gamma diagonal')
    return MappingProxyType({
        'verified': True,
        'coefficient': coefficient,
        'scientific_result_sha256': scientific_sha,
        'certificate_sha256': hashlib.sha256(certificate_bytes).hexdigest(),
        'row_manifest_sha256': manifest_sha,
        'fresh_cluster_evaluations': fresh_count,
        'resumed_cluster_evaluations': resumed_count,
        'invocation_nonce': invocation_nonce,
    })

def run_one_click_construction():
    preflight_output = run_verified(['--geometry-preflight'], capture_output=True)
    if type(preflight_output) is not str or not preflight_output.rstrip().endswith(
        PREFLIGHT_MARKER
    ):
        raise RuntimeError('Bound geometry preflight did not reach its exact PASS marker')
    print(preflight_output, end='' if preflight_output.endswith('\n') else '\n')
    preflight_json = preflight_output[:preflight_output.rfind(PREFLIGHT_MARKER)].strip()
    preflight = strict_json_bytes(
        preflight_json.encode('utf-8'), 'preflight certificate'
    )
    if (
        preflight.get('total_exact_cluster_evaluations') != 609
        or preflight.get('physics_contractions_run') != 0
        or preflight.get('necessary_not_sufficient') is not True
        or preflight.get('candidate_manifest_sha256')
        != AUTHORIZED_CANDIDATE_MANIFEST_SHA256
        or preflight.get('candidate_coverage_certificate_sha256')
        != AUTHORIZED_CANDIDATE_CERTIFICATE_SHA256
        or preflight.get('preflight_sha256') != AUTHORIZED_PREFLIGHT_SHA256
        or set(preflight.get('roots', {})) != {'0', '1', '2'}
    ):
        raise RuntimeError('Preflight certificate fixed gates changed')
    stable_preflight = dict(preflight)
    for key in ('preflight_sha256', 'elapsed_seconds', 'python_tracemalloc_peak_bytes'):
        stable_preflight.pop(key, None)
    if hashlib.sha256(canonical_bytes(stable_preflight)).hexdigest() != (
        AUTHORIZED_PREFLIGHT_SHA256
    ):
        raise RuntimeError('Preflight payload does not reproduce its reviewed SHA256')
    for pol, expected_sha in AUTHORIZED_SUPPORT_SHA256.items():
        row = preflight['roots'][pol]
        if row.get('cluster_count') != 203 or row.get('support_sha256') != expected_sha:
            raise RuntimeError(f'Polarization {pol} candidate closure changed')
    auth_context_payload = {
        'schema': AUTH_SCHEMA,
        'runtime_script_sha256': EXPECTED_SCRIPT_SHA256,
        'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
        'candidate_coverage_certificate_sha256': (
            AUTHORIZED_CANDIDATE_CERTIFICATE_SHA256
        ),
        'candidate_preflight_sha256': AUTHORIZED_PREFLIGHT_SHA256,
        'physical_cluster_evaluations': 609,
        'source_authority_sha256': preflight['source_authority_sha256'],
    }
    auth_context = hashlib.sha256(
        canonical_bytes(auth_context_payload)
    ).hexdigest()
    print('AUTOMATIC_TRIALITY_CANDIDATE_PREFLIGHT_PASS_609_NO_PHYSICS')
    if not RUN_COMPLETE_PHYSICAL_SWEEP:
        print('AUDIT_ONLY_COMPLETE_NO_PHYSICS')
        return None
    if re.fullmatch(r'[A-Za-z0-9._-]+', RUN_INSTANCE_LABEL) is None:
        raise RuntimeError('RUN_INSTANCE_LABEL must be a simple version label')
    if PERSIST_PHYSICAL_RUN_TO_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        run_dir = Path('/content/drive/MyDrive/HODGE_SU3_EXACT_M4')
    else:
        run_dir = Path('/content/HODGE_SU3_EXACT_M4')
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / (
        f'HODGE_SU3_EXACT_MARKED_CLUSTER_M4_{RUN_INSTANCE_LABEL}_CHECKPOINT.sqlite'
    )
    certificate_path = run_dir / (
        f'HODGE_SU3_EXACT_MARKED_CLUSTER_M4_{RUN_INSTANCE_LABEL}_CERTIFICATE.json'
    )
    if certificate_path.exists() and not checkpoint_path.exists():
        raise RuntimeError(
            'A certificate exists without its authenticated checkpoint; '
            'choose a NEW RUN_INSTANCE_LABEL and preserve the old artifact.'
        )
    if checkpoint_path.exists():
        metadata = read_checkpoint_metadata_without_mutation(checkpoint_path)
        if (
            metadata['schema'] != 'HODGE-SU3-PHASE3-AUTHENTICATED-SQLITE-v3'
            or metadata['runtime_script_sha256'] != EXPECTED_SCRIPT_SHA256
            or metadata['candidate_manifest_sha256']
            != AUTHORIZED_CANDIDATE_MANIFEST_SHA256
            or metadata['authentication_context_sha256'] != auth_context
        ):
            raise RuntimeError(
                'Checkpoint belongs to another build; choose a NEW RUN_INSTANCE_LABEL'
            )
        salt_hex = require_hex64(metadata['resume_salt_hex'], 'resume salt')
        run_id = require_hex64(metadata['resume_run_id'], 'resume run id')
        recovery_secret = automatic_recovery_secret(RUN_INSTANCE_LABEL)
        auth_key = derive_resume_key(
            recovery_secret, salt_hex, run_id, auth_context
        )
        immutable_meta = {
            'schema': 'HODGE-SU3-PHASE3-AUTHENTICATED-SQLITE-v3',
            'configuration_sha256': require_hex64(
                metadata['configuration_sha256'], 'configuration SHA'
            ),
            'runtime_script_sha256': EXPECTED_SCRIPT_SHA256,
            'candidate_manifest_sha256': AUTHORIZED_CANDIDATE_MANIFEST_SHA256,
            'authentication_context_sha256': auth_context,
            'resume_salt_hex': salt_hex,
            'resume_run_id': run_id,
        }
        if not hmac.compare_digest(
            metadata['key_verifier_hmac_sha256'],
            resume_hmac(auth_key, 'checkpoint-header-verifier', immutable_meta),
        ):
            auth_key = b''
            recovery_secret = None
            raise RuntimeError(
                'Checkpoint authentication failed. The DB and prior PASS were not changed. '
                'Use a NEW RUN_INSTANCE_LABEL to start a separate run.'
            )
        print('AUTHENTICATED_RESUME_HEADER_PASS')
    else:
        if checkpoint_path.with_name(checkpoint_path.name + '-journal').exists():
            raise RuntimeError(
                'Orphan journal exists at a new path; choose a NEW RUN_INSTANCE_LABEL'
            )
        salt_hex = secrets.token_hex(32)
        run_id = secrets.token_hex(32)
        recovery_secret = automatic_recovery_secret(RUN_INSTANCE_LABEL)
        auth_key = derive_resume_key(
            recovery_secret, salt_hex, run_id, auth_context
        )
    recovery_secret = None
    invocation_nonce = secrets.token_hex(32)
    auth_bundle = {
        'key': auth_key,
        'resume_salt_hex': salt_hex,
        'resume_run_id': run_id,
        'authentication_context_sha256': auth_context,
        'invocation_nonce': invocation_nonce,
    }
    try:
        certificate_bytes = run_verified(
            [
                '--run-phase3-physical',
                '--authorized-cluster-evaluations', '609',
                '--authorized-candidate-certificate-sha256',
                AUTHORIZED_CANDIDATE_CERTIFICATE_SHA256,
                '--checkpoint', str(checkpoint_path),
                '--output', str(certificate_path),
            ],
            auth_bundle=auth_bundle,
            expect_authenticated_certificate=True,
        )
        verified = verify_authenticated_result(
            certificate_bytes, certificate_path, checkpoint_path, auth_key,
            salt_hex, run_id, auth_context, invocation_nonce,
        )
    finally:
        auth_bundle.clear()
        auth_key = b''
    print('AUTHENTICATED_609_ROW_MANIFEST_PASS')
    print('fresh_cluster_evaluations=', verified['fresh_cluster_evaluations'])
    print('resumed_cluster_evaluations=', verified['resumed_cluster_evaluations'])
    print('PASS_TARGET_BLIND_M4_SEALED_AND_INDEPENDENTLY_VERIFIED')
    print('scientific_result_sha256=', verified['scientific_result_sha256'])
    print('certificate_sha256=', verified['certificate_sha256'])
    terminal_token = object()
    class CurrentRunVerifiedConstruction:
        __slots__ = ('__payload', '__token', '__consumed')
        def __init__(self, payload, token):
            if token is not terminal_token or type(payload) is not MappingProxyType:
                raise RuntimeError('Invalid current-run terminal latch construction')
            self.__payload = payload
            self.__token = token
            self.__consumed = False
        def consume_for_terminal_hamer(self):
            if self.__token is not terminal_token or self.__consumed:
                raise RuntimeError('Current-run terminal latch is invalid or already consumed')
            self.__consumed = True
            return self.__payload
    return CurrentRunVerifiedConstruction(verified, terminal_token)

CURRENT_RUN_VERIFIED_CONSTRUCTION = run_one_click_construction()


## Terminal-only historical comparison

Only after the target-blind certificate, all 609 authenticated rows, the current invocation nonce, the scientific identity, and the exact Drive copy have been independently verified does the next cell introduce Hamer's reported decimal. It reports agreement at the published 13-decimal precision; it never demands impossible exact equality and never shifts the computed coefficient.

In [ ]:
if not RUN_COMPLETE_PHYSICAL_SWEEP:
    print('HAMER_TERMINAL_COMPARISON_SKIPPED_AUDIT_ONLY')
elif not hasattr(
    CURRENT_RUN_VERIFIED_CONSTRUCTION, 'consume_for_terminal_hamer'
):
    raise RuntimeError(
        'No authenticated physical construction returned in this Run All'
    )
else:
    verified_terminal_payload = (
        CURRENT_RUN_VERIFIED_CONSTRUCTION.consume_for_terminal_hamer()
    )
    if (
        type(verified_terminal_payload) is not MappingProxyType
        or verified_terminal_payload.get('verified') is not True
    ):
        raise RuntimeError('Current-run authenticated terminal latch is malformed')
    sealed_m4 = verified_terminal_payload['coefficient']
    if type(sealed_m4) is not Fraction:
        raise RuntimeError('Verified m4 is not an exact Fraction')
    hamer_h4_reported = Fraction('-0.0968932328773')
    hamer_reported_decimal_places = 13
    hamer_rounding_half_width = Fraction(
        1, 2 * 10**hamer_reported_decimal_places
    )
    sealed_h4 = sealed_m4 / 8
    h4_difference = sealed_h4 - hamer_h4_reported
    matches_reported_precision = (
        abs(h4_difference) <= hamer_rounding_half_width
    )
    marker = (
        'HAMER_TERMINAL_MATCH_AT_REPORTED_PRECISION'
        if matches_reported_precision
        else 'HAMER_TERMINAL_MISMATCH_AT_REPORTED_PRECISION'
    )
    print(marker)
    print('sealed_m4=', sealed_m4)
    print('sealed_h4=', sealed_h4)
    print('hamer_h4_reported=', hamer_h4_reported)
    print('hamer_reported_precision_half_width=', hamer_rounding_half_width)
    print('sealed_h4_minus_hamer_reported=', h4_difference)
    print(
        'scientific_result_sha256=',
        verified_terminal_payload['scientific_result_sha256'],
    )
    print(
        'authenticated_certificate_sha256=',
        verified_terminal_payload['certificate_sha256'],
    )
